### This is the multi agent system building 
1. Policy agent
2. Logistic agent
3. Booking tool

Include router and llm judge also

### 1. Import essential libraries and configs 

In [7]:
import os
import sys
import re
import json

from typing import List, Dict, Any
from pathlib import Path

sys.path.append("..")

from utils import get_api_key, get_model_configuration, get_parameter_configuration, LLM
from tools.web_search import WebSearchTool , get_web_search_tool
from tools.booking import get_booking_tool
from rag.retriever import MedicalKnowladgeRetriever

ROOT_DIR = Path.cwd() if (Path.cwd() / "data" / "medical_guides").exists() else Path.cwd().parent
DOCS_DIR = ROOT_DIR / "data" / "medical_guides"

In [3]:
llm = LLM()
web_search = WebSearchTool()
booking_tool = get_booking_tool()
retriever = MedicalKnowladgeRetriever(docs_dir=str(DOCS_DIR))

### 2. Implement Policy Agent

In [ ]:
class PolicyAgent:
    """
    This is for Internal knowladge: RAG  
    """
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
        self.name = "Policy Agent"

    def process(self, query):
        """
        query : This is user's message
        based on the user's query find the proper answer from RAG internal
        """
        context = self.retriever.get_context(query, max_k=3)

        system_prompt = """
        You are a medical policy specialist,
        Use internal knowladge for answering to the questions about procedure, guidelines, and clinical protocols.
        Be specific and cite sources. 
        """.strip()

        prompt = f"""
        query : {query}
        Internal knowladge : {context}
        Provide a focused answer about medical policies and procedures.
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)

        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "internal_rag"
        }

        

### 3. Implement Logistic Agent

In [6]:
class LogisticAgent:
    """
    Specialist for operational logistics using web search.
    """
    def __init__(self, llm, web_search):
        self.llm = llm
        self.web_search = web_search
        self.name = "Logistic Agent"

    def process(self, query):
        """
        Provide answer to user's query from searching web 
        """
        search_result = self.web_search.search(query, max_results=3)
        search_text = self.web_search.format_results(search_result)

        system_prompt = """
        You are a healthcare logistics specialist.
        Use web search results to answer questions about hours, contacts, locations, and current operations.
        Include source URLs.
        """.strip()

        prompt = f"""
        Query: {query}
        Web Search Results:{search_text}
        Provide a focused answer about operational logistics.
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)

        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "web_search"
        }

### 4.Implement Router Agent

In [8]:
class Router:
    """
    Identifies intent and extracts patient identifier for booking lookups.
    """
    def __init__(self, llm):
        self.llm = llm

    def route(self, query):
        """
        Based on the query Classify intent (policy, logistics, booking) and extract patient identifier if booking-related.
        """
        system_prompt = """
        You are a router for a healthcare multi-agent system.
        Classify the user's intent and extract any patient identifier they mention.
        Intent:
            - policy: Questions about medical procedures, guidelines, clinical protocols, what to bring for a condition, preparation (use internal knowledge/RAG).
            - logistics: Questions about hospital hours, locations, contacts, operational info (use web search).
            - booking: Questions about the user's own appointment(s)—when, where, with whom, status, notes (use booking lookup by patient name/phone/email).
        Extract patient identifier only when booking is true: patient_name (e.g. "John Silva"), patient_phone, or patient_email from the message. Use null for missing.
        Respond with ONLY a single JSON object, no markdown, no explanation. Keys: policy (boolean), logistics (boolean), booking (boolean), patient_name (string or null), patient_phone (string or null), patient_email (string or null).
        """.strip()

        prompt = f"User query: {query}"
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        text = (response.get("response") or "").strip()

        if "```" in text:
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```\s*$", "", text)
        try:
            out = json.loads(text)
        except json.JSONDecodeError:
            out = {"policy": True, "logistics": True, "booking": False, "patient_name": None, "patient_phone": None, "patient_email": None}
        for key in ("policy", "logistics", "booking"):
            out[key] = bool(out.get(key))
        for key in ("patient_name", "patient_phone", "patient_email"):
            v = out.get(key)
            out[key] = (v.strip() if isinstance(v, str) and v.strip() else None) or None
        return out


### 5. LLM Judge

In [9]:
class Judge:
    """
    This is referes to merges result from multiple agent 
    """
    def __init__(self, llm):
        self.llm = llm

    def merge(self, query, agent_results):
        """ Merge agent response into final answer """

        merged_info = ""
        for result in agent_results:
            merged_info += f"\n\n{result['agent']} ({result['source_type']}):\n{result['response']}"

        system_prompt = """
        You are a judge synthesizing information from specialist agents.
        Create a comprehensive, well-organized final answer.
        Give priority to internal RAG for medical info, web search for operational info.
        Always end with: "This is educational information; verify with the hospital / consult your clinician."
        """.strip()

        prompt = f"""
        User Query: {query}
        Specialist Agent Responses: {merged_info}
        Synthesize these into a clear, comprehensive final answer.
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)

        return {
            "final_answer": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"]
        }


### 6. Implement Multi Agent System

In [ ]:
class MultiAgentSystem: